In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
lemma = WordNetLemmatizer()
ps = PorterStemmer()
import re


from scipy.spatial import distance
from scipy.spatial import minkowski_distance
from scipy.spatial.distance import cosine


from transformers import pipeline
import os

import gensim

from sklearn.pipeline import make_pipeline



[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sophie/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
def text_process(reviews):  #FOR EMBEDDINGS 
    for i  in range(0, reviews['review_body'].count()):
        try:
            # print(i)
            review_body=reviews.loc[i, ('review_body')]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
           
            review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
            tokens= word_tokenize(review_body)
            tokens = [w.lower()  for w in tokens ]
            tokens = [w for w in tokens if not w in stop_words]
            tokens = [w for w in tokens if w.isalpha()]
            tokens = [lemma.lemmatize(w) for w in tokens]
           # tokens = [ps.stem(w) for w in tokens]
            reviews.loc[i, ('review_body_process')]=' '.join(tokens)
           
       
        except:
           print (review_body)
       
       
        if i%10000==0:
           print ("\n text_process:   We are at i=", str(i))
       
    return reviews





In [18]:

class WordVecVectorizer(object):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = 300
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in texts.split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for texts in X
        ])




def check_against_word2vec_model(list_topics, word2vec_model):
    for i  in range(0, len(list_topics) ):
       tokens= word_tokenize(list_topics[i])
       tokens = [w for w in tokens if w in word2vec_model.key_to_index ]
       list_topics[i]=' '.join(tokens)
       return list_topics





In [49]:
def df2emd(word2vec_model, N_rewiews, column_name):
    word2vec_model_embeddings = WordVecVectorizer(word2vec_model)

    word2vec_model_embeddings_ave_one_review_list=[]
    embed_only=pd.DataFrame()
    
    for i in range(0,len(N_rewiews[column_name]) ) : 
    #for i in range(0,len(N_rewiews['review_body_process']) ) : 
        #list_words=[N_rewiews['review_body_process'][i]]
        list_words=[N_rewiews[column_name][i]]

        list_words=check_against_word2vec_model(list_words, word2vec_model)
        word2vec_embeddings_one_review=word2vec_model_embeddings.transform(list_words)
        word2vec_model_embeddings_ave_one_review_list.append(word2vec_embeddings_one_review)

    embed_only=pd.DataFrame(np.concatenate(word2vec_model_embeddings_ave_one_review_list))



    #return word2vec_model_embeddings_ave_one_review_list
    
    return embed_only

In [7]:
#Electronics Dataset:

fileElectr='amazon_reviews_us_Electronics_v1_00.tsv'
df=pd.read_csv(fileElectr, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])

In [9]:
n_samples=1000


N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=1900)
N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=1900)


samplesize=n_samples*2

N_rewiews=N_rewiews.append(N_rewiew2)
N_rewiews=text_process(N_rewiews.reset_index())





/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_25721/3968476197.py:10: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews=N_rewiews.append(N_rewiew2)



 text_process:   We are at i= 0


In [27]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix


from sklearn.model_selection import train_test_split



target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive



In [15]:
from sklearn.model_selection import train_test_split

def emb_svc(embed_only, N_rewiews):

    X_train, X_test, y_train, y_test = train_test_split(embed_only, N_rewiews['star_rating'], test_size=0.3, random_state=156)


    clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)

    clf.score(X_test, y_test)
    y_test.value_counts()
    print(clf.score(X_test, y_test))

    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

    print(confusion_matrix(y_test, y_pred))



In [16]:
file_embeddings_fast='crawl-300d-2M.vec'
file_embeddings_concept="numberbatch-en.txt"
file_embeddings_glove='glove-wiki-gigaword-300.txt'
file_embeddings_glove_s='glove-wiki-gigaword-50.txt'

In [13]:
word2vec_model_fast = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_fast) 
print(word2vec_model_fast.vector_size)


300


In [20]:
embed_only_fast_BP=df2emd(word2vec_model_fast, N_rewiews, "review_body_process")

In [28]:
emb_svc(embed_only_fast_BP, N_rewiews)

0.8633333333333333
                 precision    recall  f1-score   support

0 = rating of 1   0.857627  0.863481  0.860544       293
1 = rating of 5   0.868852  0.863192  0.866013       307

       accuracy                       0.863333       600
      macro avg   0.863240  0.863337  0.863279       600
   weighted avg   0.863371  0.863333  0.863342       600

[[253  40]
 [ 42 265]]


In [29]:
embed_only_fast_B=df2emd(word2vec_model_fast, N_rewiews, "review_body")

In [30]:
emb_svc(embed_only_fast_B, N_rewiews)

0.8833333333333333
                 precision    recall  f1-score   support

0 = rating of 1   0.885813  0.873720  0.879725       293
1 = rating of 5   0.881029  0.892508  0.886731       307

       accuracy                       0.883333       600
      macro avg   0.883421  0.883114  0.883228       600
   weighted avg   0.883365  0.883333  0.883310       600

[[256  37]
 [ 33 274]]


In [31]:
word2vec_model_concept = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_concept) 
print(word2vec_model_concept.vector_size)


embed_only_concept_BP=df2emd(word2vec_model_concept, N_rewiews, "review_body_process")
embed_only_concept_B=df2emd(word2vec_model_concept, N_rewiews, "review_body")


300


In [32]:
emb_svc(embed_only_concept_BP, N_rewiews)
emb_svc(embed_only_concept_B, N_rewiews)

0.8516666666666667
                 precision    recall  f1-score   support

0 = rating of 1   0.846939  0.849829  0.848382       293
1 = rating of 5   0.856209  0.853420  0.854812       307

       accuracy                       0.851667       600
      macro avg   0.851574  0.851625  0.851597       600
   weighted avg   0.851682  0.851667  0.851672       600

[[249  44]
 [ 45 262]]
0.83
                 precision    recall  f1-score   support

0 = rating of 1   0.852399  0.788396  0.819149       293
1 = rating of 5   0.811550  0.869707  0.839623       307

       accuracy                       0.830000       600
      macro avg   0.831974  0.829051  0.829386       600
   weighted avg   0.831498  0.830000  0.829625       600

[[231  62]
 [ 40 267]]


In [33]:
word2vec_model_glove = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_glove)
print(word2vec_model_glove.vector_size)



embed_only_glove_BP=df2emd(word2vec_model_glove, N_rewiews, "review_body_process")
embed_only_glove_B=df2emd(word2vec_model_glove, N_rewiews, "review_body")



300


In [34]:
emb_svc(embed_only_glove_BP, N_rewiews)
emb_svc(embed_only_glove_B, N_rewiews)

0.835
                 precision    recall  f1-score   support

0 = rating of 1   0.810897  0.863481  0.836364       293
1 = rating of 5   0.861111  0.807818  0.833613       307

       accuracy                       0.835000       600
      macro avg   0.836004  0.835649  0.834989       600
   weighted avg   0.836590  0.835000  0.834956       600

[[253  40]
 [ 59 248]]
0.81
                 precision    recall  f1-score   support

0 = rating of 1   0.795380  0.822526  0.808725       293
1 = rating of 5   0.824916  0.798046  0.811258       307

       accuracy                       0.810000       600
      macro avg   0.810148  0.810286  0.809992       600
   weighted avg   0.810492  0.810000  0.810021       600

[[241  52]
 [ 62 245]]


In [43]:
N_rewiews.columns

Index(['index', 'marketplace', 'customer_id', 'review_id', 'product_id',
       'product_parent', 'product_title', 'product_category', 'star_rating',
       'helpful_votes', 'total_votes', 'vine', 'verified_purchase',
       'review_headline', 'review_body', 'review_date', 'review_body_process'],
      dtype='object')

In [44]:
def text_process_review_headline(reviews):  #FOR EMBEDDINGS 
    for i  in range(0, reviews['review_headline'].count()):
        try:
            # print(i)
            review_body=reviews.loc[i, ('review_headline')]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
           
            review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
            tokens= word_tokenize(review_body)
            tokens = [w.lower()  for w in tokens ]
            tokens = [w for w in tokens if not w in stop_words]
            tokens = [w for w in tokens if w.isalpha()]
            tokens = [lemma.lemmatize(w) for w in tokens]
           # tokens = [ps.stem(w) for w in tokens]
            reviews.loc[i, ('review_headline_process')]=' '.join(tokens)
           
       
        except:
           print (review_body)
       
       
        if i%10000==0:
           print ("\n text_process:   We are at i=", str(i))
       
    return reviews






In [45]:
N_rewiews=text_process_review_headline(N_rewiews)

N_rewiews.columns


 text_process:   We are at i= 0


Index(['index', 'marketplace', 'customer_id', 'review_id', 'product_id',
       'product_parent', 'product_title', 'product_category', 'star_rating',
       'helpful_votes', 'total_votes', 'vine', 'verified_purchase',
       'review_headline', 'review_body', 'review_date', 'review_body_process',
       'review_headline_process'],
      dtype='object')

In [50]:
embed_only_fast_HP=df2emd(word2vec_model_fast, N_rewiews, "review_headline")
embed_only_fast_H=df2emd(word2vec_model_fast, N_rewiews, "review_headline_process")

In [52]:
emb_svc(embed_only_fast_HP, N_rewiews)
emb_svc(embed_only_fast_H, N_rewiews)



0.855
                 precision    recall  f1-score   support

0 = rating of 1   0.845638  0.860068  0.852792       293
1 = rating of 5   0.864238  0.850163  0.857143       307

       accuracy                       0.855000       600
      macro avg   0.854938  0.855116  0.854967       600
   weighted avg   0.855155  0.855000  0.855018       600

[[252  41]
 [ 46 261]]
0.8266666666666667
                 precision    recall  f1-score   support

0 = rating of 1   0.816054  0.832765  0.824324       293
1 = rating of 5   0.837209  0.820847  0.828947       307

       accuracy                       0.826667       600
      macro avg   0.826631  0.826806  0.826636       600
   weighted avg   0.826878  0.826667  0.826690       600

[[244  49]
 [ 55 252]]


In [53]:
embed_only_concept_HP=df2emd(word2vec_model_concept, N_rewiews, "review_headline_process")
embed_only_concept_H=df2emd(word2vec_model_concept, N_rewiews, "review_headline")

emb_svc(embed_only_concept_HP, N_rewiews)
emb_svc(embed_only_concept_H, N_rewiews)








0.8316666666666667
                 precision    recall  f1-score   support

0 = rating of 1   0.820000  0.839590  0.829680       293
1 = rating of 5   0.843333  0.824104  0.833608       307

       accuracy                       0.831667       600
      macro avg   0.831667  0.831847  0.831644       600
   weighted avg   0.831939  0.831667  0.831690       600

[[246  47]
 [ 54 253]]
0.7
                 precision    recall  f1-score   support

0 = rating of 1   0.792746  0.522184  0.629630       293
1 = rating of 5   0.656020  0.869707  0.747899       307

       accuracy                       0.700000       600
      macro avg   0.724383  0.695946  0.688764       600
   weighted avg   0.722788  0.700000  0.690144       600

[[153 140]
 [ 40 267]]


In [54]:
embed_only_glove_HP=df2emd(word2vec_model_glove, N_rewiews, "review_headline_process")
embed_only_glove_H=df2emd(word2vec_model_glove, N_rewiews, "review_headline")

emb_svc(embed_only_glove_HP, N_rewiews)
emb_svc(embed_only_glove_H, N_rewiews)






0.8233333333333334
                 precision    recall  f1-score   support

0 = rating of 1   0.808581  0.836177  0.822148       293
1 = rating of 5   0.838384  0.811075  0.824503       307

       accuracy                       0.823333       600
      macro avg   0.823482  0.823626  0.823325       600
   weighted avg   0.823830  0.823333  0.823353       600

[[245  48]
 [ 58 249]]
0.665
                 precision    recall  f1-score   support

0 = rating of 1   0.709091  0.532423  0.608187       293
1 = rating of 5   0.639474  0.791531  0.707424       307

       accuracy                       0.665000       600
      macro avg   0.674282  0.661977  0.657805       600
   weighted avg   0.673470  0.665000  0.658963       600

[[156 137]
 [ 64 243]]


In [ ]:
#FastText with unprocessed body and unprocessed headline produce best results

In [56]:
#embed_only_fast_HP
#embed_only_fast_B

embed_combined=embed_only_fast_B.join(embed_only_fast_HP, lsuffix='_caller', rsuffix='_other')

embed_combined.head(2)

,0_caller,1_caller,2_caller,3_caller,4_caller,5_caller,6_caller,7_caller,8_caller,9_caller,...,290_other,291_other,292_other,293_other,294_other,295_other,296_other,297_other,298_other,299_other
0,-0.010403,-0.063589,-0.030153,-0.073472,0.025708,0.032458,-0.007578,0.044956,-0.040789,0.018031,...,-0.046375,0.040000,0.054738,-0.060075,0.036063,-0.156525,0.039750,-0.137062,-0.054550,0.018938
1,-0.017315,0.007123,-0.041455,-0.078637,-0.087770,0.033452,-0.058645,-0.031260,-0.046363,0.013523,...,-0.084700,-0.008533,-0.047967,-0.041167,-0.152000,0.031833,0.003667,0.141400,0.022133,0.114100


In [60]:
embed_combined.shape

(2000, 600)

In [57]:

embed_only_fast_B.head(2)

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,-0.010403,-0.063589,-0.030153,-0.073472,0.025708,0.032458,-0.007578,0.044956,-0.040789,0.018031,...,-0.005167,0.030714,0.056650,-0.093389,-0.037575,-0.112233,0.032256,0.028661,-0.072689,-0.005867
1,-0.017315,0.007123,-0.041455,-0.078637,-0.087770,0.033452,-0.058645,-0.031260,-0.046363,0.013523,...,-0.017533,0.027030,-0.030302,-0.057590,-0.050950,-0.043297,-0.001945,0.044050,-0.098903,0.017910


In [58]:
embed_only_fast_HP.head(2)



,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,-0.010363,-0.125762,0.024113,-0.069025,-0.078387,0.012425,-0.053850,0.00815,-0.112838,0.022588,...,-0.046375,0.040000,0.054738,-0.060075,0.036063,-0.156525,0.039750,-0.137062,-0.054550,0.018938
1,-0.082600,0.077833,-0.066900,-0.039767,-0.266200,0.185167,0.166367,0.05890,0.039200,-0.147833,...,-0.084700,-0.008533,-0.047967,-0.041167,-0.152000,0.031833,0.003667,0.141400,0.022133,0.114100


In [61]:
emb_svc(embed_combined, N_rewiews)



0.9233333333333333
                 precision    recall  f1-score   support

0 = rating of 1   0.921502  0.921502  0.921502       293
1 = rating of 5   0.925081  0.925081  0.925081       307

       accuracy                       0.923333       600
      macro avg   0.923292  0.923292  0.923292       600
   weighted avg   0.923333  0.923333  0.923333       600

[[270  23]
 [ 23 284]]


In [72]:
embed_combined=embed_only_fast_B.join(embed_only_fast_H, lsuffix='_caller', rsuffix='_other')
emb_svc(embed_combined, N_rewiews)



0.8933333333333333
                 precision    recall  f1-score   support

0 = rating of 1   0.896194  0.883959  0.890034       293
1 = rating of 5   0.890675  0.902280  0.896440       307

       accuracy                       0.893333       600
      macro avg   0.893435  0.893120  0.893237       600
   weighted avg   0.893370  0.893333  0.893312       600

[[259  34]
 [ 30 277]]


In [62]:
N_rewiews.head(1)

,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,review_body_process,review_headline_process
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,1,N,Y,"Does it keep time, not really.","I have had the product for just under a month,...",2013-12-14,product month opened time perfectly set fallen...,keep time really


In [64]:
N_rewiews.shape

(2000, 18)

In [68]:
from transformers import pipeline
import os


#distilbart-cnn-12-6
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")
model = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6")
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer) #sshleifer/distilbart-cnn-12-6 (https://huggingface.co/sshleifer/distilbart-cnn-12-6)

for i in range (0, len(N_rewiews.review_body)):

    text= N_rewiews.review_body[i]
    if len(text) > 512:
        text = text[:512]
    
    
    summary_text = summarizer(text, max_length=15, min_length=5, do_sample=False)[0]['summary_text']
    N_rewiews.loc[i, ('distilbart_T')]=summary_text
    
    text=N_rewiews.review_headline[i] + ". "+ N_rewiews.review_body[i]
    if len(text) > 512:
            text = text[:512]
    
    
    summary_text = summarizer(text, max_length=10, min_length=5, do_sample=False)[0]['summary_text']
    N_rewiews.loc[i, ('distilbart_TP')]=summary_text
    


    

Your max_length is set to 15, but you input_length is only 5. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=2)
Your max_length is set to 10, but you input_length is only 8. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 14. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)
Your max_length is set to 15, but you input_length is only 10. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)
Your max_length is set to 15, but you input_length is only 12. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 15, but you input_length is only 5. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=2)
Your max_length is set to 10, but you input_length is only 8. You might consider decreasing max_l

Your max_length is set to 15, but you input_length is only 12. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 15, but you input_length is only 14. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)
Your max_length is set to 15, but you input_length is only 11. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)
Your max_length is set to 15, but you input_length is only 7. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 15, but you input_length is only 12. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 15, but you input_length is only 8. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 7. You might consider decreasing max_

Your max_length is set to 15, but you input_length is only 11. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)
Your max_length is set to 15, but you input_length is only 11. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)
Your max_length is set to 15, but you input_length is only 6. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 10, but you input_length is only 9. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 9. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 6. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 10, but you input_length is only 9. You might consider decreasing max_le

Your max_length is set to 15, but you input_length is only 6. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 10, but you input_length is only 9. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 5. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=2)
Your max_length is set to 10, but you input_length is only 8. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 7. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 15, but you input_length is only 5. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=2)
Your max_length is set to 10, but you input_length is only 8. You might consider decreasing max_leng

Your max_length is set to 15, but you input_length is only 11. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)
Your max_length is set to 15, but you input_length is only 10. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)
Your max_length is set to 15, but you input_length is only 9. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 8. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 3. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=1)
Your max_length is set to 10, but you input_length is only 6. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 15, but you input_length is only 8. You might consider decreasing max_le

Your max_length is set to 10, but you input_length is only 9. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 8. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 13. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 15, but you input_length is only 14. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)
Your max_length is set to 15, but you input_length is only 3. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=1)
Your max_length is set to 10, but you input_length is only 6. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 15, but you input_length is only 12. You might consider decreasing max_l

Your max_length is set to 15, but you input_length is only 11. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)
Your max_length is set to 15, but you input_length is only 7. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 15, but you input_length is only 13. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 15, but you input_length is only 6. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 10, but you input_length is only 9. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 15, but you input_length is only 14. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)
Your max_length is set to 15, but you input_length is only 3. You might consider decreasing max_l

In [69]:
N_rewiews.head(1)


,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,review_body_process,review_headline_process,distilbart_T,distilbart_TP
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,1,N,Y,"Does it keep time, not really.","I have had the product for just under a month,...",2013-12-14,product month opened time perfectly set fallen...,keep time really,I have had the product for just under a month...,I have had the product for just


In [70]:
embed_only_glove_T=df2emd(word2vec_model_glove, N_rewiews, "distilbart_T")
emb_svc(embed_only_glove_T, N_rewiews)

embed_only_concept_T=df2emd(word2vec_model_concept, N_rewiews, "distilbart_T")
emb_svc(embed_only_concept_T, N_rewiews)

embed_only_fast_T=df2emd(word2vec_model_fast, N_rewiews, "distilbart_T")
emb_svc(embed_only_fast_T, N_rewiews)



0.7016666666666667
                 precision    recall  f1-score   support

0 = rating of 1   0.686275  0.716724  0.701169       293
1 = rating of 5   0.717687  0.687296  0.702163       307

       accuracy                       0.701667       600
      macro avg   0.701981  0.702010  0.701666       600
   weighted avg   0.702347  0.701667  0.701677       600

[[210  83]
 [ 96 211]]
0.7166666666666667
                 precision    recall  f1-score   support

0 = rating of 1   0.704319  0.723549  0.713805       293
1 = rating of 5   0.729097  0.710098  0.719472       307

       accuracy                       0.716667       600
      macro avg   0.716708  0.716824  0.716638       600
   weighted avg   0.716997  0.716667  0.716704       600

[[212  81]
 [ 89 218]]
0.755
                 precision    recall  f1-score   support

0 = rating of 1   0.757042  0.733788  0.745234       293
1 = rating of 5   0.753165  0.775244  0.764045       307

       accuracy                       0.755000 

In [71]:
embed_only_glove_TP=df2emd(word2vec_model_glove, N_rewiews, "distilbart_TP")
emb_svc(embed_only_glove_TP, N_rewiews)


embed_only_concept_TP=df2emd(word2vec_model_concept, N_rewiews, "distilbart_TP")
emb_svc(embed_only_concept_TP, N_rewiews)

embed_only_fast_TP=df2emd(word2vec_model_fast, N_rewiews, "distilbart_TP")
emb_svc(embed_only_fast_TP, N_rewiews)


0.67
                 precision    recall  f1-score   support

0 = rating of 1   0.666667  0.648464  0.657439       293
1 = rating of 5   0.673016  0.690554  0.681672       307

       accuracy                       0.670000       600
      macro avg   0.669841  0.669509  0.669556       600
   weighted avg   0.669915  0.670000  0.669838       600

[[190 103]
 [ 95 212]]
0.6916666666666667
                 precision    recall  f1-score   support

0 = rating of 1   0.703008  0.638225  0.669052       293
1 = rating of 5   0.682635  0.742671  0.711388       307

       accuracy                       0.691667       600
      macro avg   0.692821  0.690448  0.690220       600
   weighted avg   0.692583  0.691667  0.690714       600

[[187 106]
 [ 79 228]]
0.76
                 precision    recall  f1-score   support

0 = rating of 1   0.750842  0.761092  0.755932       293
1 = rating of 5   0.768977  0.758958  0.763934       307

       accuracy                       0.760000       600
     

In [ ]:
#Pegasus
#https://huggingface.co/google/pegasus-xsum

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")
model = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-xsum")
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)


for i in range (0, len(N_rewiews.review_body)):
    text=N_rewiews.review_body[i]
    if len(text) > 512:
        text = text[:512]
    summary_text = summarizer(text, max_length=10, min_length=5, do_sample=False)[0]['summary_text']
    N_rewiews.loc[i, ('pegasus')]=summary_text
    
    text=N_rewiews.review_headline[i] + ". "+ N_rewiews.review_body[i]
    if len(text) > 512:
        text = text[:512]
    summary_text = summarizer(text, max_length=10, min_length=5, do_sample=False)[0]['summary_text']
    N_rewiews.loc[i, ('pegasus_TP')]=summary_text
    



Your max_length is set to 10, but you input_length is only 4. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=2)
Your max_length is set to 10, but you input_length is only 7. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 10, but you input_length is only 9. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)
Your max_length is set to 10, but you input_length is only 4. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=2)
Your max_length is set to 10, but you input_length is only 7. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 10, but you input_length is only 3. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=1)
Your max_length is set to 10, but you input_length is only 6. You might consider decreasing max_leng

In [ ]:
embed_only_glove_T_pegasus=df2emd(word2vec_model_glove, N_rewiews, "pegasus")
emb_svc(embed_only_glove_T_pegasus, N_rewiews)

embed_only_concept_T_pegasus=df2emd(word2vec_model_concept, N_rewiews, "pegasus")
emb_svc(embed_only_concept_T_pegasus, N_rewiews)

embed_only_fast_T_pegasus=df2emd(word2vec_model_fast, N_rewiews, "pegasus")
emb_svc(embed_only_fast_T_pegasus, N_rewiews)





In [ ]:
embed_only_glove_TP_pegasus=df2emd(word2vec_model_glove, N_rewiews, "pegasus_TP")
emb_svc(embed_only_glove_TP_pegasus, N_rewiews)

embed_only_concept_TP_pegasus=df2emd(word2vec_model_concept, N_rewiews, "pegasus_TP")
emb_svc(embed_only_concept_TP_pegasus, N_rewiews)

embed_only_fast_TP_pegasus=df2emd(word2vec_model_fast, N_rewiews, "pegasus_TP")
emb_svc(embed_only_fast_TP_pegasus, N_rewiews)



In [ ]:
#https://huggingface.co/ml6team/distilbart-tos-summarizer-tosdr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("ml6team/distilbart-tos-summarizer-tosdr")
model = AutoModelForSeq2SeqLM.from_pretrained("ml6team/distilbart-tos-summarizer-tosdr")
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)


for i in range (0, len(N_rewiews.review_body)):
    text=N_rewiews.review_body[i] 
    if len(text) > 512:
        text = text[:512]
    summary_text = summarizer(text, max_length=10, min_length=5, do_sample=False)[0]['summary_text']
    N_rewiews.loc[i, ('Dtos')]=summary_text
    
    text=N_rewiews.review_headline[i] + ". "+ N_rewiews.review_body[i]
    if len(text) > 512:
        text = text[:512]
    summary_text = summarizer(text, max_length=10, min_length=5, do_sample=False)[0]['summary_text']
    N_rewiews.loc[i, ('Dtos_TP')]=summary_text
    
        

In [ ]:
embed_only_glove_T_Dtos=df2emd(word2vec_model_glove, N_rewiews, "Dtos")
emb_svc(embed_only_glove_T_Dtos, N_rewiews)

embed_only_concept_T_Dtos=df2emd(word2vec_model_concept, N_rewiews, "Dtos")
emb_svc(embed_only_concept_T_Dtos, N_rewiews)

embed_only_fast_T_Dtos=df2emd(word2vec_model_fast, N_rewiews, "Dtos")
emb_svc(embed_only_fast_T_Dtos, N_rewiews)






In [ ]:
embed_only_glove_TP_Dtos=df2emd(word2vec_model_glove, N_rewiews, "Dtos_TP")
emb_svc(embed_only_glove_TP_Dtos, N_rewiews)

embed_only_concept_TP_Dtos=df2emd(word2vec_model_concept, N_rewiews, "Dtos_TP")
emb_svc(embed_only_concept_TP_Dtos, N_rewiews)

embed_only_fast_TP_Dtos=df2emd(word2vec_model_fast, N_rewiews, "Dtos_TP")
emb_svc(embed_only_fast_TP_Dtos, N_rewiews)


